<a href="https://colab.research.google.com/github/PoonamSonawane2109/Fake-Job-Detection/blob/main/Fake_Job_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets pandas numpy scikit-learn -q

print("✅ All libraries installed!")

✅ All libraries installed!


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torch
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset

print("✅ All imports done!")
print(f"GPU available: {torch.cuda.is_available()}")
# Should print: GPU available: True

✅ All imports done!
GPU available: True


In [3]:
import pandas as pd

url = "https://huggingface.co/datasets/victor/real-or-fake-fake-jobposting-prediction/resolve/main/fake_job_postings.csv"
df = pd.read_csv(url)
print(df.head())
print(df.shape)
print(df.info())

   job_id                                      title            location  \
0       1                           Marketing Intern    US, NY, New York   
1       2  Customer Service - Cloud Video Production      NZ, , Auckland   
2       3    Commissioning Machinery Assistant (CMA)       US, IA, Wever   
3       4          Account Executive - Washington DC  US, DC, Washington   
4       5                        Bill Review Manager  US, FL, Fort Worth   

  department salary_range                                    company_profile  \
0  Marketing          NaN  We're Food52, and we've created a groundbreaki...   
1    Success          NaN  90 Seconds, the worlds Cloud Video Production ...   
2        NaN          NaN  Valor Services provides Workforce Solutions th...   
3      Sales          NaN  Our passion for improving quality of life thro...   
4        NaN          NaN  SpotSource Solutions LLC is a Global Human Cap...   

                                         description  \
0  Foo

In [4]:
# fraudulent = 1 means FAKE, fraudulent = 0 means REAL

print("Fake vs Real job counts:")
print(df['fraudulent'].value_counts())

print(f"\nTotal jobs: {len(df)}")
print(f"Fake jobs: {df['fraudulent'].sum()}")
print(f"Real jobs: {(df['fraudulent']==0).sum()}")

Fake vs Real job counts:
fraudulent
0    17014
1      866
Name: count, dtype: int64

Total jobs: 17880
Fake jobs: 866
Real jobs: 17014


In [5]:
#  Read one actual fake job to understand the data

fake_example = df[df['fraudulent'] == 1].iloc[0]

print("=== FAKE JOB EXAMPLE ===")
print(f"Title:       {fake_example['title']}")
print(f"Description: {str(fake_example['description'])[:300]}...")

print("\n=== REAL JOB EXAMPLE ===")
real_example = df[df['fraudulent'] == 0].iloc[0]
print(f"Title:       {real_example['title']}")
print(f"Description: {str(real_example['description'])[:300]}...")

=== FAKE JOB EXAMPLE ===
Title:       IC&E Technician
Description: IC&amp;E Technician | Bakersfield, CA Mt. PosoPrincipal Duties and Responsibilities: Calibrates, tests, maintains, troubleshoots, and installs all power plant instrumentation, control systems and electrical equipment.Performs maintenance on motor control centers, motor operated valves, generators, e...

=== REAL JOB EXAMPLE ===
Title:       Marketing Intern
Description: Food52, a fast-growing, James Beard Award-winning online food community and crowd-sourced and curated recipe hub, is currently interviewing full- and part-time unpaid interns to work in a small team of editors, executives, and developers in its New York City headquarters.Reproducing and/or repackagi...


In [6]:
# Quick check on your loaded dataset

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFake vs Real:")
print(df['fraudulent'].value_counts())
print("\nMissing values per column:")
print(df.isnull().sum())

Shape: (17880, 18)

Columns: ['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']

Fake vs Real:
fraudulent
0    17014
1      866
Name: count, dtype: int64

Missing values per column:
job_id                     0
title                      0
location                 346
department             11547
salary_range           15012
company_profile         3308
description                1
requirements            2696
benefits                7212
telecommuting              0
has_company_logo           0
has_questions              0
employment_type         3471
required_experience     7050
required_education      8105
industry                4903
function                6455
fraudulent                 0
dtype: int64


In [8]:
#combine columns into one text field

# Combine title + description + requirements into one text
# Why? DistilBERT reads ONE text input — we merge the most useful columns

def combine_text(row):
    parts = [
        str(row['title']),
        str(row['description']),
        str(row['requirements']),
        str(row['company_profile'])
    ]
    # Join all parts, replace 'nan' (empty cells) with empty string
    combined = ' '.join([p if p != 'nan' else '' for p in parts])
    return combined.strip()

df['text'] = df.apply(combine_text, axis=1)

print("✅ Text column created!")
print("\nExample fake job text (first 400 chars):")
print(df[df['fraudulent']==1].iloc[0]['text'][:400])#

✅ Text column created!

Example fake job text (first 400 chars):
IC&E Technician IC&amp;E Technician | Bakersfield, CA Mt. PosoPrincipal Duties and Responsibilities: Calibrates, tests, maintains, troubleshoots, and installs all power plant instrumentation, control systems and electrical equipment.Performs maintenance on motor control centers, motor operated valves, generators, excitation equipment and motors.Performs preventive, predictive and corrective mainte


In [9]:
#Keep only text + label columns, drop empty rows

df_clean = df[['text', 'fraudulent']].copy()

# Rename fraudulent to label (cleaner name)
df_clean = df_clean.rename(columns={'fraudulent': 'label'})

# Remove rows where text is empty
df_clean = df_clean[df_clean['text'].str.len() > 10]

# Reset index
df_clean = df_clean.reset_index(drop=True)

print(f"✅ Clean dataset ready!")
print(f"Total rows: {len(df_clean)}")
print(f"Fake: {df_clean['label'].sum()}")
print(f"Real: {(df_clean['label']==0).sum()}")

✅ Clean dataset ready!
Total rows: 17880
Fake: 866
Real: 17014


In [10]:
#split into train and test sets

# Cell 10: Split data — 80% for training, 20% for testing
# Think of it like: study 80% of questions, test on remaining 20%

train_df, test_df = train_test_split(
    df_clean,
    test_size=0.2,        # 20% goes to test
    random_state=42,     # fixed seed so results are same every run
    stratify=df_clean['label']  # keeps fake/real ratio balanced
)

print(f"✅ Split done!")
print(f"Training rows : {len(train_df)}")
print(f"Testing rows  : {len(test_df)}")
print(f"\nTraining fake jobs : {train_df['label'].sum()}")
print(f"Training real jobs : {(train_df['label']==0).sum()}")

✅ Split done!
Training rows : 14304
Testing rows  : 3576

Training fake jobs : 693
Training real jobs : 13611
